# Failure Memory (Observability MCP)

Build the failure taxonomy and cross-session learning system.

## Learning Objectives
- Understand the failure category taxonomy
- Record and categorize failures
- Use failure history to generate improvement hints
- Enable cross-session learning from past failures
- See how failure memory drives the harness Reflect → Plan feedback loop

In [ ]:
import os
import sys
import httpx
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
env_path = os.path.join('..', '.env')
load_dotenv(env_path, override=True)

_VERIFY_SSL = os.getenv("VERIFY_SSL", "true").lower() not in ("false", "0", "no")
_http_client = None if _VERIFY_SSL else httpx.Client(verify=False, timeout=httpx.Timeout(300.0))

from harness.failure import FailureCategory, FailureLog
print("Failure module loaded")

## 1. Failure Taxonomy

The taxonomy categorizes research failures into actionable types:

In [ ]:
# Display all failure categories
print("Failure Taxonomy:")
print("=" * 50)

categories = {
    "Content Quality": [
        FailureCategory.INSUFFICIENT_DEPTH,
        FailureCategory.MISSING_CITATIONS,
        FailureCategory.HALLUCINATION,
        FailureCategory.OFF_TOPIC,
        FailureCategory.REPETITIVE,
        FailureCategory.POOR_STRUCTURE,
    ],
    "Retrieval": [
        FailureCategory.LOW_RELEVANCE,
        FailureCategory.NO_RESULTS,
        FailureCategory.WRONG_CONTEXT,
    ],
    "System": [
        FailureCategory.TIMEOUT,
        FailureCategory.AGENT_ERROR,
        FailureCategory.MCP_ERROR,
        FailureCategory.LLM_ERROR,
        FailureCategory.TOKEN_LIMIT,
    ],
    "Verification": [
        FailureCategory.QUALITY_BELOW_THRESHOLD,
        FailureCategory.CITATION_INVALID,
        FailureCategory.FACT_CHECK_FAILED,
    ],
}

for group, cats in categories.items():
    print(f"\n  {group}:")
    for cat in cats:
        print(f"    - {cat.value}")

## 2. Recording Failures

In [ ]:
# Create a failure log and record some failures
log = FailureLog()

# Simulate failures across iterations
log.record("session_1", 1, FailureCategory.INSUFFICIENT_DEPTH, 
           "Report only covers surface-level information")
log.record("session_1", 1, FailureCategory.MISSING_CITATIONS,
           "3 claims have no source reference")
log.record("session_1", 2, FailureCategory.LOW_RELEVANCE,
           "Search returned off-topic results for sub-query")

print(f"Recorded {len(log.get_failures('session_1'))} failures for session_1")
print(f"Categories: {log.get_failure_categories('session_1')}")

## 3. Generating Improvement Hints

In [ ]:
# Get improvement hints based on accumulated failures
hints = log.get_improvement_hints("session_1")
print("Improvement hints for next iteration:")
print(hints)
print("\n(These hints are passed to the planner in the next iteration)")

## 4. Cross-Session Learning

In [ ]:
# Persist failures to PostgreSQL
log.persist("session_1")
print("Failures persisted to PostgreSQL")

# Load past failure patterns (cross-session)
past_patterns = log.load_past_failures(limit=10)
if past_patterns:
    print("\nPast failure patterns (across all sessions):")
    for p in past_patterns:
        print(f"  {p['category']} — occurred {p['count']} time(s)")
        if p.get('examples'):
            print(f"    Example: {p['examples'][0][:80]}")
else:
    print("\nNo past failure patterns found (first run)")

## 5. How Learning Feeds Back

The learning loop:
1. **Session N fails** → failures categorized and stored
2. **Session N+1 starts** → Context Layer loads past failure patterns
3. **Planner receives hints** → generates plan that avoids known issues
4. **Better first-iteration quality** → fewer iterations needed

Over time, the system learns to avoid common failure modes.

## Summary

Failure memory:
- **Taxonomy**: 17 categorized failure types across 4 groups
- **Hints**: Automatic improvement suggestions from failure patterns
- **Persistence**: PostgreSQL storage enables cross-session learning
- **Feedback loop**: Past failures improve future sessions